# MobiFall Visualisation

This notebook loads the smartphone fall dataset through the grouped-trial
parser used by the fall pipeline.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CWD = Path.cwd().resolve()
for candidate in (CWD, *CWD.parents):
    if (candidate / "pipeline").exists() and (candidate / "analysis").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repo root from the current working directory.")

for extra_path in (REPO_ROOT, REPO_ROOT / "analysis"):
    extra_str = str(extra_path)
    if extra_str not in sys.path:
        sys.path.insert(0, extra_str)

import notebook_utils as nb_utils

nb_utils.configure_matplotlib()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [ ]:
from pipeline.ingest import load_mobifall

DATASET_PATH = REPO_ROOT / "data" / "raw" / "MOBIACT_Dataset" / "MobiFall_Dataset_v2.0"
LOAD_FULL_DATASET = False
MAX_TRIALS = None if LOAD_FULL_DATASET else 40

if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

df = load_mobifall(DATASET_PATH, max_files=MAX_TRIALS)
df = nb_utils.add_vector_magnitudes(df)
df["activity_code"] = df["session_id"].astype(str).str.split(":", n=1).str[0]

print(f"Loaded {len(df):,} rows from {DATASET_PATH}")
display(df.head())


In [ ]:
profile_df = nb_utils.dataset_profile(df)
loader_notes_df = pd.DataFrame({"loader_note": df.attrs.get("loader_notes", [])})
label_raw_df = nb_utils.count_table(df, "label_raw", top_n=10)
label_mapped_df = nb_utils.count_table(df, "label_mapped", top_n=10)
activity_code_df = nb_utils.count_table(df, "activity_code", top_n=12)

trial_sensor_df = (
    df.groupby("session_id", dropna=False, sort=False)[["has_gyro", "has_orientation"]]
    .max()
    .reset_index()
)
gyro_presence_df = (
    trial_sensor_df["has_gyro"].astype(str).value_counts().rename_axis("has_gyro").reset_index(name="count")
)
orientation_presence_df = (
    trial_sensor_df["has_orientation"].astype(str).value_counts().rename_axis("has_orientation").reset_index(name="count")
)

display(profile_df)
display(loader_notes_df if not loader_notes_df.empty else pd.DataFrame({"loader_note": ["No loader notes recorded"]}))
display(label_raw_df)
display(activity_code_df)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
nb_utils.plot_count_bars(label_raw_df, "label_raw", ax=axes[0, 0], title="Raw labels", color="#3a86ff")
nb_utils.plot_count_bars(label_mapped_df, "label_mapped", ax=axes[0, 1], title="Mapped fall labels", color="#ff006e")
nb_utils.plot_count_bars(activity_code_df, "activity_code", ax=axes[1, 0], title="Trial activity codes", color="#8338ec")
nb_utils.plot_count_bars(gyro_presence_df, "has_gyro", ax=axes[1, 1], title="Trials with gyroscope stream", color="#2a9d8f")
fig.tight_layout()


In [ ]:
_ = nb_utils.plot_signal_histograms(df, dataset_label="MobiFall")


In [ ]:
duration_df = nb_utils.session_duration_table(df)
display(duration_df.head())

fig, ax = plt.subplots(figsize=(10, 4))
duration_df.boxplot(column="duration_s", by="label_mapped", ax=ax, grid=False)
ax.set_title("Trial duration by mapped label")
ax.set_xlabel("label_mapped")
ax.set_ylabel("duration_s")
fig.suptitle("")
fig.tight_layout()


In [ ]:
example_seq = nb_utils.pick_representative_sequence(df, preferred_label="fall", min_rows=128)
display(example_seq[["subject_id", "session_id", "timestamp", "ax", "ay", "az", "gx", "gy", "gz", "label_mapped"]].head(12))
_ = nb_utils.plot_sequence_axes(example_seq, title="Representative MobiFall fall sequence")
